# SensorFM — build data (Kaggle, CPU, free)

Settings: **Accelerator: None**, **Internet: On**. Add Secrets (Add-ons → Secrets): `GH_TOKEN` (read-only GitHub token for the private repo) and optionally `HF_TOKEN`.

Output: `/kaggle/working/{downstream,shards/main,results}` → after the run, *Save Version*, then create a **private Dataset** named `sensorfm-data` from this notebook's output.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['GH_TOKEN'] = s.get_secret('GH_TOKEN')
try:
    os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
except Exception:
    print('no HF_TOKEN secret (optional)')

In [ ]:
%%bash
set -e
cd /kaggle/working
rm -rf sensorfm-src && git clone -q https://$GH_TOKEN@github.com/DhrumilPanchal10/sensorfm.git sensorfm-src
cd sensorfm-src && git log --oneline -1
WORK=/kaggle/working TMP_CACHE=/tmp/hf bash kaggle/build_data.sh 2>&1 | grep -v Warning

In [ ]:
import json
m = json.load(open('/kaggle/working/shards/main/manifest.json'))
for split in ('train', 'val'):
    sp = m['splits'][split]
    print(split, sp['n_series'], 'series', f"{sp['n_obs']:,}", 'obs', sp['obs_per_domain'])
print('decontaminated:', m['build']['decontaminated_series'])
print('manifest sha256:', m['manifest_sha256'])
# Keep the output small: drop the source checkout before saving the version.
import shutil; shutil.rmtree('/kaggle/working/sensorfm-src', ignore_errors=True)